In [ ]:
import cv2

# Load the video
video_path = 'VIDEO.avi'  # Replace with your video file path
cap = cv2.VideoCapture(video_path)

# Create background subtractor objects
fgbg_mog2 = cv2.createBackgroundSubtractorMOG2()
fgbg_knn = cv2.createBackgroundSubtractorKNN()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Resize frame for faster processing (optional)
    frame_resized = cv2.resize(frame, (640, 480))

    # 1. MOG2 Background Subtraction
    fgmask_mog2 = fgbg_mog2.apply(frame_resized)

    # 2. KNN Background Subtraction
    fgmask_knn = fgbg_knn.apply(frame_resized)

    # Show the results
    cv2.imshow('Original Video', frame_resized)
    cv2.imshow('MOG2 Background Subtraction', fgmask_mog2)
    cv2.imshow('KNN Background Subtraction', fgmask_knn)

    # Press 'q' to quit
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np

# Open a video capture
cap = cv2.VideoCapture('VIDEO.avi')  # Use 0 for webcam, or replace with video path

# --- Running Average Background Subtraction ---
avg = None  # Initialize average frame

# --- MOG2 Background Subtraction ---
mog2 = cv2.createBackgroundSubtractorMOG2()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Resize for faster processing (optional)
    frame = cv2.resize(frame, (640, 480))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)

    # --- Running Average ---
    if avg is None:
        avg = gray.copy().astype("float")
        continue

    cv2.accumulateWeighted(gray, avg, 0.05)  # 0.05 is the learning rate
    running_avg = cv2.convertScaleAbs(avg)
    diff = cv2.absdiff(gray, running_avg)
    _, running_avg_mask = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)

    # --- MOG2 ---
    mog2_mask = mog2.apply(frame)

    # Display results
    cv2.imshow("Original", frame)
    cv2.imshow("Running Avg Mask", running_avg_mask)
    cv2.imshow("MOG2 Mask", mog2_mask)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()